# Module 10 — FMEA (Failure Mode and Effects Analysis)

Part of the Root-Cause Analysis Playbook. Program and progress: [CURRICULUM.md](CURRICULUM.md).

## Why you're here: what this module is for

- Every earlier module looked back at an incident: why did it happen, why did nothing stop it. FMEA (Failure Mode and Effects Analysis) looks forward: **how could it fail next, how bad would it be, and would we notice?**
- It ranks the ways a system can fail, so the team knows which ones to watch most closely.
- In DMAIC (Define, Measure, Analyze, Improve, Control), it is a **Control** tool: its output is the control plan, the list of what to monitor so the fixes stay fixed.

**What this module produces:**
1. An FMEA for a data platform, with every rating taken from a year of incident data on chosen scales, not from opinion.
2. A check of the classic ranking number, the RPN (Risk Priority Number), against the real loss. It ranks one serious failure mode last.
3. The modern alternative, Action Priority, and a control plan for the failure modes it marks High.

## From the curriculum

> ## <u>Module 10 — FMEA (Failure Mode and Effects Analysis)</u>
>
> **DMAIC phase:** Control (ranking what could still fail, and turning it into
> the control plan).
>
> **Concepts**
> - Failure mode, effect, cause, current control
> - The three ratings: Severity, Occurrence, Detection (1–10)
> - RPN (Risk Priority Number = S × O × D) and its known pitfalls (equal RPNs
>   hiding very different risks)
> - AIAG-VDA (Automotive Industry Action Group / German Association of the
>   Automotive Industry) Action Priority as the modern replacement
> - From FMEA to control plan: each high-priority mode gets a monitor, a limit,
>   an owner and a response
>
> **Why it matters**
> FMEA is the forward-looking twin of root-cause analysis: instead of "why did
> it fail?" it asks "how could it fail next, and would we notice?" It closes the
> DMAIC loop.
>
> **Worked example**
> Fresh, simulated: an FMEA for the data platform from Module 2. Occurrence is
> rated from the incident-log frequencies and Detection from the logged
> time-to-detect, so the ratings come from data, not opinion. Checked against
> the real loss, the RPN ranks the warehouse outage last although it is the
> second-largest loss. Action Priority marks it High.
>
> **Resources**
> - [How to Perform FMEA: Methodology and Worked Examples](https://reliamag.com/guides/how-to-perform-fmea/)
>   (Reliable Media, ~20 min, verified): the steps, the S/O/D scales, the RPN's
>   problems and Action Priority (described, not reproduced: the official table
>   is in the AIAG-VDA handbook).
>
> **Exercise type:** re-rate a failure mode after a fix, and check whether its
> Action Priority changes. Fully worked.
>
> **Interview angle:** "How do you prioritise which risks to monitor?"

## Lesson

### The parts

- **Failure mode:** one way the system can fail (*"an upstream team changes a column without warning"*).
- **Effect:** what the failure does to the people who rely on the system.
- **Cause:** why the failure mode happens.
- **Current control:** what is in place today to prevent it or catch it.

### The three ratings, each 1 to 10

- **Severity (S):** how bad the effect is. 10 = worst.
- **Occurrence (O):** how often it happens. 10 = most often.
- **Detection (D):** how long the failure goes unnoticed after it starts. 10 = longest. (Note the direction: a **high** D is **bad**.) In many FMEAs, D asks whether the failure is caught before it reaches the user. This module rates the closest thing the data has: minutes to detect.

### RPN, and why it misleads

The classic ranking number is the **RPN**: RPN = S × O × D, from 1 to 1,000. Its known problems:
- **The ratings are ranks, not amounts.** Severity 8 is not "twice as bad" as severity 4 (a chosen example). Multiplying ranks gives a number with no real meaning.
- **Equal RPNs can hide very different risks.** S = 9, O = 3, D = 1 gives 27. So does S = 3, O = 3, D = 3. The first is a severe failure that is rare and noticed fast: these are the warehouse outage's ratings in this module's example. The second is a minor failure on every count (a chosen example, to show the problem).
- **A low Occurrence or a low Detection can hide a high Severity.** A rare, quickly noticed failure gets a small RPN however bad it is.

### Action Priority: the modern replacement

The AIAG-VDA handbook (AIAG = Automotive Industry Action Group, VDA = the German Association of the Automotive Industry), in its 2019 edition, replaced the RPN with **Action Priority (AP)**: a lookup table that gives each S, O, D combination a priority of **High**, **Medium** or **Low**. It looks at **Severity first**, then Occurrence, then Detection, so severity weighs more than the other two.

### From FMEA to control plan

For each high-priority failure mode, the **control plan** names:
- **what to monitor:** a measure that shows the failure mode starting
- **the limit:** the value that triggers a response
- **the owner:** who watches it
- **the response:** what to do when the limit is crossed

### Who does what

The failure modes, their causes and the controls come from the people who run the system: engineers, operators, product owners. The analyst brings the data that turns S, O and D from opinions into measurements on agreed scales, and checks whether the ranking matches the real loss.

## Worked example: an FMEA for the data platform

**Data choice: reuse, simulated.** The same simulated year of incidents as Module 2 (`common.simulate_incidents`, fixed seed): one row per incident on a company's data platform, with its category, minutes to resolve, users affected, user-hours lost and minutes to detect. Every number in the simulation is a chosen example. Reusing it means S, O and D can all be rated from data, and the ranking can be checked against the real loss.

**The failure modes:** the incident categories. One is left out: "Other", because it is a bucket of mixed incidents, not one way of failing.

**Terms in the category names:**
- **warehouse:** the central database where the platform's data is stored
- **upstream:** the teams and systems that send data into the platform
- **schema:** the list of columns in a table or file, with each column's name and type
- **seed:** the starting number of the random generator. Each seed gives a different simulated year, with the same settings.

In [1]:
import numpy as np
import pandas as pd
import common
from common import show

inc = common.simulate_incidents()
inc = inc[inc["category"] != "Other"]
g = inc.groupby("category").agg(incidents=("incident_id", "size"),
                                 uh_median=("user_hours_lost", "median"),
                                 detect_median=("minutes_to_detect", "median"),
                                 uh_total=("user_hours_lost", "sum"))
g = g.sort_values("uh_median", ascending=False)
print("What is in the data? (simulated, 2025)")
print(f"  incidents, without \"Other\" = {len(inc)}")
print(f"  failure modes = {len(g)}")
print()
print("Each failure mode, from the incident log:\n")
show(pd.DataFrame({"failure mode": g.index, "incidents in 2025": g["incidents"].to_numpy(),
                   "user-hours lost, median incident": g["uh_median"].map("{:,.1f}".format).to_numpy(),
                   "minutes to detect, median incident": g["detect_median"].map("{:,.0f}".format).to_numpy()}))
print()
print("user-hours lost = minutes to resolve ÷ 60 × users affected")
print("median incident = the middle one when sorted")

What is in the data? (simulated, 2025)
  incidents, without "Other" = 614
  failure modes = 7

Each failure mode, from the incident log:

failure mode                  incidents in 2025  user-hours lost, median incident  minutes to detect, median incident
Warehouse outage                              5                           2,913.8                                   5
Upstream schema change                       31                           1,213.3                                 195
Bad data (duplicates, nulls)                 35                             163.3                                 592
Dashboard bug                                56                              47.0                                 269
Late upstream file                          252                               9.9                                  21
Failed scheduled job                        154                               3.4                                  10
Permission error                    

**Observation:** the failure modes differ by factors of 10 and more. A warehouse outage costs 2,913.8 user-hours in its median incident, a permission error 1.2. Late upstream files happen 252 times a year, warehouse outages 5 times.

### Step 1 — Turn the data into ratings

Each rating needs a scale: which range of the data gets which number from 1 to 10.

**The scales (chosen, because):** the data spans factors of 10 and more, so each step multiplies the range by a fixed amount instead of adding to it. The starting points and the multipliers are chosen so the observed values spread across the 1 to 10 range:
- **Severity:** user-hours lost in the median incident. Rating 1 = up to 1 user-hour, and each step = 3 times more.
- **Occurrence:** incidents per year. Rating 1 = up to 2 incidents, and each step = 2 times more.
- **Detection:** minutes to detect the median incident. Rating 1 = up to 5 minutes, and each step = 2 times longer.

In [2]:
edges = {"S": [1, 3, 9, 27, 81, 243, 729, 2187, 6561],
         "O": [2, 4, 8, 16, 32, 64, 128, 256, 512],
         "D": [5, 10, 20, 40, 80, 160, 320, 640, 1280]}
rate = lambda v, e: 1 + sum(v > x for x in e)
units = {"S": "user-hours", "O": "incidents a year", "D": "minutes"}
rows = []
for r in range(1, 11):
    row = {"rating": r}
    for k, e in edges.items():
        lo = 0 if r == 1 else e[r - 2]
        row[f"{k} ({units[k]})"] = f"X ≤ {e[0]:,}" if r == 1 else (f"X > {lo:,}" if r == 10 else f"{lo:,} < X ≤ {e[r - 1]:,}")
    rows.append(row)
print("The three scales:\n")
show(pd.DataFrame(rows))

The three scales:

rating  S (user-hours)     O (incidents a year)  D (minutes)
     1  X ≤ 1              X ≤ 2                 X ≤ 5
     2  1 < X ≤ 3          2 < X ≤ 4             5 < X ≤ 10
     3  3 < X ≤ 9          4 < X ≤ 8             10 < X ≤ 20
     4  9 < X ≤ 27         8 < X ≤ 16            20 < X ≤ 40
     5  27 < X ≤ 81        16 < X ≤ 32           40 < X ≤ 80
     6  81 < X ≤ 243       32 < X ≤ 64           80 < X ≤ 160
     7  243 < X ≤ 729      64 < X ≤ 128          160 < X ≤ 320
     8  729 < X ≤ 2,187    128 < X ≤ 256         320 < X ≤ 640
     9  2,187 < X ≤ 6,561  256 < X ≤ 512         640 < X ≤ 1,280
    10  X > 6,561          X > 512               X > 1,280


In [3]:
g["S"] = g["uh_median"].apply(lambda v: rate(v, edges["S"]))
g["O"] = g["incidents"].apply(lambda v: rate(v, edges["O"]))
g["D"] = g["detect_median"].apply(lambda v: rate(v, edges["D"]))
g["RPN"] = g["S"] * g["O"] * g["D"]
g = g.sort_values("RPN", ascending=False)
g["RPN rank"] = range(1, len(g) + 1)
print("The FMEA, ranked by RPN = S × O × D:\n")
show(pd.DataFrame({"failure mode": g.index, "S": g["S"].to_numpy(), "O": g["O"].to_numpy(), "D": g["D"].to_numpy(),
                   "RPN": g["RPN"].to_numpy(), "RPN rank": g["RPN rank"].to_numpy()}))

The FMEA, ranked by RPN = S × O × D:

failure mode                  S  O  D  RPN  RPN rank
Bad data (duplicates, nulls)  6  6  8  288         1
Upstream schema change        8  5  7  280         2
Dashboard bug                 5  6  7  210         3
Late upstream file            4  8  4  128         4
Permission error              2  7  4   56         5
Failed scheduled job          3  8  2   48         6
Warehouse outage              9  3  1   27         7


**Observation:** by RPN, bad data (288) and upstream schema changes (280) come first. The warehouse outage comes last, with an RPN of 27, although its severity is the highest of all (S = 9).

### Step 2 — Does the RPN rank the failure modes by their real loss?

The real loss of a failure mode over the year is the total user-hours lost across all its incidents. The log has it, so the RPN ranking can be checked against it:

In [4]:
g["loss rank"] = g["uh_total"].rank(ascending=False).astype(int)
show(pd.DataFrame({"failure mode": g.index, "RPN": g["RPN"].to_numpy(), "RPN rank": g["RPN rank"].to_numpy(),
                   "user-hours lost in 2025": g["uh_total"].map("{:,.0f}".format).to_numpy(), "loss rank": g["loss rank"].to_numpy()}))
w = {k: (int(v) if k != "uh_total" else v) for k, v in g.loc["Warehouse outage", ["S", "O", "D", "RPN", "RPN rank", "loss rank", "uh_total"]].items()}
print()
print("The warehouse outage:")
print(f"  RPN = {w['S']} × {w['O']} × {w['D']} = {w['RPN']}, ranked {w['RPN rank']} of {len(g)}")
print(f"  user-hours lost in 2025 = {w['uh_total']:,.0f}, ranked {w['loss rank']} of {len(g)}")
print(f"  share of all user-hours lost = {w['uh_total']:,.0f} ÷ {g['uh_total'].sum():,.0f} = {w['uh_total'] / g['uh_total'].sum():.1%}")

failure mode                  RPN  RPN rank  user-hours lost in 2025  loss rank
Bad data (duplicates, nulls)  288         1                    8,052          3
Upstream schema change        280         2                   45,668          1
Dashboard bug                 210         3                    3,115          4
Late upstream file            128         4                    3,104          5
Permission error               56         5                      141          7
Failed scheduled job           48         6                      632          6
Warehouse outage               27         7                   26,590          2

The warehouse outage:
  RPN = 9 × 3 × 1 = 27, ranked 7 of 7
  user-hours lost in 2025 = 26,590, ranked 2 of 7
  share of all user-hours lost = 26,590 ÷ 87,302 = 30.5%


**Observation:** the RPN puts the warehouse outage last, 7th of 7. By real loss it is 2nd: 26,590 user-hours in 2025, 30.5% of everything lost.

**Why they disagree:**
- The total loss adds up the user-hours of every incident, so it grows with how often and how bad. It has no detection term.
- The RPN multiplies in D as well. The warehouse outage has the highest severity (S = 9), but it happened only 5 times (O = 3) and is noticed within minutes (D = 1). The low O and D pull its RPN down to 27.
- Noticing an outage quickly does not make it smaller: the warehouse is still down for everyone until it is fixed.

Does this hold beyond one simulated year? The same check on ten draws of the same simulation, each with a different seed (ten, chosen: enough to see whether the result repeats):

In [5]:
def rpn_vs_loss(seed):
    d = common.simulate_incidents(seed)
    d = d[d["category"] != "Other"]
    t = d.groupby("category").agg(n=("incident_id", "size"), uh=("user_hours_lost", "median"),
                                  det=("minutes_to_detect", "median"), tot=("user_hours_lost", "sum"))
    rpn = t["uh"].apply(lambda v: rate(v, edges["S"])) * t["n"].apply(lambda v: rate(v, edges["O"])) * t["det"].apply(lambda v: rate(v, edges["D"]))
    return int(rpn.rank(ascending=False, method="min")["Warehouse outage"]), int(t["tot"].rank(ascending=False)["Warehouse outage"])

res = pd.DataFrame([(s, *rpn_vs_loss(s)) for s in [2026] + list(range(1, 10))], columns=["seed", "warehouse RPN rank", "warehouse loss rank"])
print("The warehouse outage's two ranks, on ten draws of the simulation (of 7 failure modes):\n")
show(res.assign(seed=["2026 (main example)"] + [str(x) for x in res["seed"].iloc[1:]]))
print()
print(f"Draws where the RPN ranks it 6th or 7th = {(res['warehouse RPN rank'] >= 6).sum()} of {len(res)}")
print(f"Draws where the loss ranks it in the top 3 = {(res['warehouse loss rank'] <= 3).sum()} of {len(res)}")

The warehouse outage's two ranks, on ten draws of the simulation (of 7 failure modes):

seed                 warehouse RPN rank  warehouse loss rank
2026 (main example)                   7                    2
1                                     6                    2
2                                     7                    2
3                                     7                    5
4                                     6                    2
5                                     7                    3
6                                     6                    2
7                                     7                    2
8                                     7                    3
9                                     7                    2

Draws where the RPN ranks it 6th or 7th = 10 of 10
Draws where the loss ranks it in the top 3 = 9 of 10


**Observation:** in all ten draws, the RPN ranks the warehouse outage 6th or 7th. In 9 of them, its real loss is in the top 3.

**Conclusion:** ranked by RPN, the failure mode that cost the second-most user-hours would be the last to get attention. This is the "low Occurrence or low Detection hides a high Severity" problem, on real numbers, and it is not luck of one draw.

### Step 3 — Action Priority

The official AIAG-VDA lookup table is in the handbook, which is not freely published. So this module uses a **simplified rule built on the handbook's principle**, Severity first, then Occurrence, then Detection (chosen, because the official table cannot be reproduced here). The cut-offs follow that principle: the higher S, the less O or D it takes to be High, and a severity of 9–10 is never Low:
- **S 9–10:** High if O ≥ 2, otherwise Medium.
- **S 7–8:** High if O ≥ 5 or D ≥ 7, otherwise Medium.
- **S 4–6:** Medium if O ≥ 7 or D ≥ 7, otherwise Low.
- **S 1–3:** Low.

**Data caveat:** summaries of the official table disagree on cases like the warehouse outage (S = 9, very rare, noticed fast): some put them at High, some lower. So the priorities in this module are this simplified rule's, not the handbook's.

In [6]:
def ap(s, o, d):
    if s >= 9:
        return "High" if o >= 2 else "Medium"
    if s >= 7:
        return "High" if (o >= 5 or d >= 7) else "Medium"
    if s >= 4:
        return "Medium" if (o >= 7 or d >= 7) else "Low"
    return "Low"

g["AP"] = [ap(s, o, d) for s, o, d in zip(g["S"], g["O"], g["D"])]
order = {"High": 0, "Medium": 1, "Low": 2}
t = g.assign(k=g["AP"].map(order)).sort_values(["k", "RPN rank"])
print("The FMEA, by Action Priority:\n")
show(pd.DataFrame({"failure mode": t.index, "S": t["S"].to_numpy(), "O": t["O"].to_numpy(), "D": t["D"].to_numpy(),
                   "AP": t["AP"].to_numpy(), "RPN rank": t["RPN rank"].to_numpy(), "loss rank": t["loss rank"].to_numpy()}))

The FMEA, by Action Priority:

failure mode                  S  O  D  AP      RPN rank  loss rank
Upstream schema change        8  5  7  High           2          1
Warehouse outage              9  3  1  High           7          2
Bad data (duplicates, nulls)  6  6  8  Medium         1          3
Dashboard bug                 5  6  7  Medium         3          4
Late upstream file            4  8  4  Medium         4          5
Permission error              2  7  4  Low            5          7
Failed scheduled job          3  8  2  Low            6          6


**Observation:** under this rule, two failure modes are High: the warehouse outage and upstream schema changes. They are also the top 2 by real loss. The warehouse outage moves from last by RPN to High, because its severity is looked at first.

**Observation:** bad data, first by RPN, is Medium: its severity (S = 6) is lower than the two High modes.

Does the warehouse outage stay High on the other draws of the simulation?

In [7]:
def warehouse_ap(seed):
    d = common.simulate_incidents(seed)
    w = d[d["category"] == "Warehouse outage"]
    s, o, dd = rate(w["user_hours_lost"].median(), edges["S"]), rate(len(w), edges["O"]), rate(w["minutes_to_detect"].median(), edges["D"])
    return len(w), s, o, dd, ap(s, o, dd)

wap = pd.DataFrame([(str(x), *warehouse_ap(x)) for x in [2026] + list(range(1, 10))], columns=["seed", "incidents in the year", "S", "O", "D", "AP"])
wap.loc[0, "seed"] = "2026 (main example)"
print("The warehouse outage's ratings and Action Priority, on ten draws of the simulation:\n")
show(wap)
print()
for lvl in ["High", "Medium", "Low"]:
    print(f"Draws with AP {lvl} = {(wap['AP'] == lvl).sum()} of {len(wap)}")

The warehouse outage's ratings and Action Priority, on ten draws of the simulation:

seed                 incidents in the year  S  O  D  AP
2026 (main example)                      5  9  3  1  High
1                                        5  8  3  2  Medium
2                                        2  9  1  2  Medium
3                                        1  9  1  2  Medium
4                                        5  9  3  2  High
5                                        3  9  2  1  High
6                                        6  9  3  2  High
7                                        3  9  2  1  High
8                                        3  8  2  1  Medium
9                                        6  8  3  1  Medium

Draws with AP High = 5 of 10
Draws with AP Medium = 5 of 10
Draws with AP Low = 0 of 10


**Observation:** the warehouse outage is High in 5 of the 10 draws and Medium in the other 5. It is never Low, unlike its RPN rank, which is 6th or 7th in every draw.

**Observation:** in the Medium draws, either its severity lands on 8 instead of 9, or it happened only 1 or 2 times, so O = 1.

**Conclusion:** Action Priority keeps the warehouse outage in the top two priority levels in every draw, where the RPN put it near the bottom. Whether it reaches High depends on where it falls on the scales: one step of S, or one step of O, flips it to Medium.

**Hypothesis:** the scale boundaries matter as much as the rule. A severity scale with different cut-offs would move some failure modes across them.

### Step 4 — The control plan

For each High failure mode in the main example: what to monitor, the limit, the owner and the response. (Proposed, as a chosen example: a real control plan is agreed with the team that runs the platform. TBD = to be decided: the owners are left to be assigned. The limits are chosen too: checking every minute and waiting for 2 failures in a row means a single short glitch alerts no one.)

| failure mode | monitor | limit | owner | response |
|---|---|---|---|---|
| Warehouse outage | a check that the warehouse answers a small test request, every minute | 2 failed checks in a row | TBD | alert the engineer on duty and post a status message to users |
| Upstream schema change | a column check: each incoming file's columns against the expected list, before loading | any column added, removed, renamed or with a changed type | TBD | stop the load and alert the data team before any table is updated |

The two controls work differently. The warehouse outage is already detected fast (D = 1), so its control is about responding fast. The schema change is detected slowly (D = 7), so its control is about catching it before it does damage.

## Exercise — Re-rate after the fix

FMEA is not done once: after an action, the failure mode is rated again. In the simulation, a schema check went live on 2025-07-01 (the July check). It catches most renamed columns before they turn into incidents. It does not catch the other kinds of schema change (a column dropped, or its type changed).

Re-rate "Upstream schema change" on the first half of 2025 (before the July check) and on the second half (after it). To compare with the yearly scale, count each half's incidents twice (a half year × 2 = a year). Does its Action Priority change?

In [8]:
sc = inc[inc["category"] == "Upstream schema change"]
halves = {"01.01-30.06 (before the July check)": sc[sc["date"] < "2025-07-01"], "01.07-31.12 (after the July check)": sc[sc["date"] >= "2025-07-01"]}
rows = []
for name, h in halves.items():
    n_year = 2 * len(h)
    uh, det = h["user_hours_lost"].median(), h["minutes_to_detect"].median()
    s, o, dd = rate(uh, edges["S"]), rate(n_year, edges["O"]), rate(det, edges["D"])
    rows.append({"half": name, "incidents in the half": len(h), "a year at this rate": n_year, "user-hours lost, median incident": f"{uh:,.1f}",
                 "minutes to detect, median incident": f"{det:,.0f}", "S": s, "O": o, "D": dd, "RPN": s * o * dd, "AP": ap(s, o, dd)})
r = pd.DataFrame(rows)
show(r)
print()
print("How much did the RPN fall?")
print(f"  {r['RPN'][0]} − {r['RPN'][1]} = {r['RPN'][0] - r['RPN'][1]}")
print(f"  {r['RPN'][0] - r['RPN'][1]} ÷ {r['RPN'][0]} = {(r['RPN'][0] - r['RPN'][1]) / r['RPN'][0]:.1%}")
print()
print("Incidents by kind of schema change, in each half:")
print()
kinds = sc.assign(half=np.where(sc["date"] < "2025-07-01", "01.01-30.06", "01.07-31.12")).groupby(["sub_cause", "half"]).size().unstack(fill_value=0)
show(kinds.rename_axis("kind of change"), index=True)
oth = kinds.drop(index="Column renamed")
print()
print("The two kinds the July check does not catch, together:")
for col in kinds.columns:
    print(f"  {col} = {' + '.join(str(v) for v in oth[col])} = {oth[col].sum()}")

half                                 incidents in the half  a year at this rate  user-hours lost, median incident  minutes to detect, median incident  S  O  D  RPN  AP
01.01-30.06 (before the July check)                     25                   50                           1,141.8                                 201  8  6  7  336  High
01.07-31.12 (after the July check)                       6                   12                           1,526.3                                 172  8  4  7  224  High

How much did the RPN fall?
  336 − 224 = 112
  112 ÷ 336 = 33.3%

Incidents by kind of schema change, in each half:



kind of change       01.01-30.06  01.07-31.12
Column dropped                 5            2
Column renamed                12            2
Column type changed            8            2

The two kinds the July check does not catch, together:
  01.01-30.06 = 5 + 8 = 13
  01.07-31.12 = 2 + 2 = 4


**Answer:**
- Occurrence falls from O = 6 (50 incidents a year at the first half's rate) to O = 4 (12 a year).
- The RPN falls from 336 to 224.
- The Action Priority stays **High**: severity is still 8, and detection is still slow (D = 7), so a schema change that gets through is still costly and still noticed late.

**Data caveat:** the second half has only 6 incidents, so its medians, and S and D with them, are fragile.

**Observation:** renamed columns, the kind the July check catches, fell from 12 to 2. The other two kinds fell too, from 13 to 4 together. In the simulation the check does not affect them, so with counts this small that fall is chance.

**Observation:** with S = 8 and D = 7, this rule gives High whatever O is. So here, Action Priority cannot show a fix that only lowers O.

**Conclusion:** in the simulation, the July check cut how often schema changes become incidents, but not how bad or how hidden the remaining ones are.

**Hypothesis:** the column check in the control plan, which covers every kind of change, would also bring D down, and with it the priority.

## Key takeaways

- **FMEA looks forward:** how could it fail, how bad would it be, would we notice? It closes the DMAIC loop with a control plan.
- **Rate from data where you can.** Here S, O and D come from a year of incidents on chosen scales, so the ratings are measurements, not opinions. The scales themselves are still a choice.
- **The RPN can hide a serious risk.** It ranked the warehouse outage last, though it cost 30.5% of all user-hours lost, because a low O and D pulled its high S down.
- **Check the ranking against the real loss.** A ranking that disagrees with the loss is a ranking to question.
- **Action Priority looks at Severity first**, so a severe failure mode is much harder to hide: the warehouse outage was High or Medium in all 10 draws. But one step on a scale can move it between High and Medium.
- **Re-rate after every fix.** The July check cut the RPN by 33.3%, but the AP stayed High: the fix lowered how often, not how bad or how hidden.
- **In DMAIC, this is Control:** the FMEA picks what to monitor, and the control plan says how.